In [1]:
import cv2
import numpy as np
import pandas as pd
import os

# === 入力ファイルパス ===
fixation_csv_path = "exported_csv/fixation_IVT/fix_df_001-001-0.csv"
aoi_image_path = "../face_aoi_project/output_aoi/1-1.jpg"

# === 出力先 ===
output_csv_path = "exported_csv/fixation_with_AOI_label_001-001-0.csv"

# === 読み込み ===
fixation_df = pd.read_csv(fixation_csv_path)
img = cv2.imread(aoi_image_path)

# === AOIマスク作成 ===
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
lower_cyan = np.array([80, 50, 50])
upper_cyan = np.array([100, 255, 255])
mask_blue = cv2.inRange(hsv, lower_cyan, upper_cyan)

contours, _ = cv2.findContours(mask_blue, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
sorted_contours = sorted(contours, key=cv2.contourArea, reverse=True)[:4]

# マスクリスト
aoi_masks = []
for contour in sorted_contours:
    mask = np.zeros(mask_blue.shape, dtype=np.uint8)
    cv2.drawContours(mask, [contour], -1, 255, thickness=cv2.FILLED)
    aoi_masks.append(mask)

# AOIを上からy座標順に並べる
aoi_info = []
for contour in sorted_contours:
    M = cv2.moments(contour)
    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        aoi_info.append({"contour": contour, "cx": cx, "cy": cy})

aoi_info_sorted = sorted(aoi_info, key=lambda d: d["cy"])
eyes = sorted(aoi_info_sorted[:2], key=lambda d: d["cx"])
nose = aoi_info_sorted[2]
mouth = aoi_info_sorted[3]

aoi_labels = ["left_eye", "right_eye", "nose", "mouth"]

# === 全注視点にラベル付け ===
labels_per_fixation = []
for _, row in fixation_df.iterrows():
    x, y = int(row["x_px"]), int(row["y_px"])
    label = "outside"

    for i, mask in enumerate(aoi_masks):
        if 0 <= x < mask.shape[1] and 0 <= y < mask.shape[0]:
            if mask[y, x] == 255:
                label = aoi_labels[i]
                break

    labels_per_fixation.append(label)

# === データフレームに列を追加 ===
fixation_df["AOI_label"] = labels_per_fixation


fixation_df
# === 保存 ===
# fixation_df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

# print("処理完了: 出力ファイル =>", output_csv_path)


,start_time,end_time,duration_ms,x_mean_deg,y_mean_deg,trial,x_px,y_px,x_norm,y_norm,AOI_label
0,1.732521e+09,1.732521e+09,204.999924,-2.767835,0.072304,0.0,842.995900,543.051263,0.439060,0.502825,nose
1,1.732521e+09,1.732521e+09,223.000050,-0.257302,0.057585,0.0,949.131523,542.430120,0.494339,0.502250,outside
2,1.732521e+09,1.732521e+09,160.000086,-0.619488,0.046685,0.0,933.831888,541.970126,0.486371,0.501824,outside
3,1.732521e+09,1.732521e+09,174.999952,-1.122158,0.159418,0.0,912.594099,546.727549,0.475309,0.506229,nose
4,1.732521e+09,1.732521e+09,175.999880,-0.144324,0.118204,0.0,953.903745,544.988287,0.496825,0.504619,outside
...,...,...,...,...,...,...,...,...,...,...,...
70,1.732521e+09,1.732521e+09,302.999973,-4.967430,-9.075564,0.0,749.648941,153.771096,0.390442,0.142381,outside
71,1.732521e+09,1.732521e+09,270.999908,-2.902228,-10.684570,0.0,837.305238,83.805438,0.436096,0.077598,outside
72,1.732521e+09,1.732521e+09,255.999804,-6.024950,-9.342390,0.0,704.564500,142.214924,0.366961,0.131680,outside
73,1.732521e+09,1.732521e+09,223.000050,-7.288560,-8.514903,0.0,650.460467,177.997863,0.338781,0.164813,outside
